# Mock experiment for simulation of cell growth using MorphoElastic framework

This section imports all the necessary libraries for the experiment.

In [1]:
from bvpy.boundary_conditions import Boundary, dirichlet
from bvpy.vforms.elasticity import StVenantKirchoffPotential
from bvpy.vforms.plasticity import MorphoElasticForm
from bvpy.vforms.plasticity import ConstantGrowth, HeterogeneousGrowth
from bvpy.gbvp import GBVP
from bvpy.utils.post_processing import SolutionExplorer
from bvpy.utils.visu_pyvista import visualize
import numpy as np
import pyvista as pv

[gvf178:120665] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.gvf178.1000/jf.0/3188916224/shared_mem_cuda_pool.gvf178 could be created.
[gvf178:120665] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 


### **Experiment Explanation**

We will simulate a mock cell growth experiment with two attached cells. The goal is to:

1. **Create mock data**: Simulate the growth of two connected cells with different growth tensors and compute their displacement and strain fields.
2. **Extract the strain-rate of each cells**: Use the computed strain fields as a proxy for the growth tensors of each cell.
3. **Make the simulation of the cell growth**: Simulate the growth using these strain-derived growth tensors and compare the results with the initial simulation.

## Create mock data

**Geometry : initial state**

This cell creates a simple geometry containing two subdomains using the `TwoRectangle` class.

- A large rectangle (`rect1`) and a smaller one (`rect2`) inside it are defined.
- The geometry is fragmented into two subdomains: `cell1` and `cell2`.


In [2]:
from bvpy.domains.abstract import OccModel, AbstractDomain

class TwoRectangle(AbstractDomain, OccModel):
    def __init__(self, L=2, H=1, h=0.5, **kwargs):
        super(TwoRectangle, self).__init__(**kwargs)
        self.geometry(L, H, h)

    def geometry(self, L, H, h):
        # Create two rectangles
        rect1 = self.factory.addRectangle(0, 0, 0, L, H) # first large rectangle
        rect2 = self.factory.addRectangle(0, 0, 0, L, h) # second small rectangle included in the first
        surfs, _ = self.factory.fragment([(2, rect1)], [(2, rect2)]) # fragment the large rectangle using the other one

        self.factory.synchronize()

        # Add labels & names to the subdomains
        self.model.addPhysicalGroup(2, [surfs[0][1]], 1) # assign label 1 to the subdomain defined by the first rectangle
        self.model.addPhysicalGroup(2, [surfs[1][1]], 2) # assign label 2 to the subdomain defined by the second rectangle
        self.model.setPhysicalName(2, 1, 'cell1') # add name to the subdomain 1
        self.model.setPhysicalName(2, 2, 'cell2') # add name to the subdomain 2

Discretize the geometry and visualize the subdomains using `pyvista`.

In [3]:
rect = TwoRectangle(L=2, H=0.5, h=0.25, cell_size=0.05, dim=2, clear=True, algorithm='Frontal-Delaunay')
rect.discretize()

pl = visualize(rect, 'domain', show_plot=False)
pl.view_xy()
pl.show()

Info    : Clearing all models and views...
Info    : Done clearing all models and views


Widget(value='<iframe src="http://localhost:36699/index.html?ui=P_0x7fec2e338a60_0&reconnect=auto" class="pyvi…

**Create an artificial grow**

Here, we:
- Fix the cells at two points on the boundary: (0,0) and (0,0.25).
- Define boundary conditions, the St. Venant-Kirchhoff potential, and morphoelastic forms with growth tensors for each cell.

In [10]:
# Fixed boundary points
bc_pts1 = dirichlet(val=[0, 0], boundary=Boundary("near(x, 0)") & Boundary("near(y, 0)"), method="pointwise")
bc_pts2 = dirichlet(val=[0, 0], boundary=Boundary("near(x, 0)") & Boundary("near(y, 0.25)"), method="pointwise")
bc = [bc_pts1, bc_pts2]

# Define Potential Energy and Plastic Form
potential_energy = StVenantKirchoffPotential(young=0.0001, poisson=0)  # Stiffness parameters
plastic_vform = MorphoElasticForm(potential_energy=potential_energy, F_g=np.identity(2), source=[0, 0])

# Growth tensors
cell1_x_growth = 0.05
cell2_y_growth = 0.3

growth_cell1 = ConstantGrowth(dg=np.array([[-cell1_x_growth, 0], [0, 0]]))  # x-growth for cell1
growth_cell2 = ConstantGrowth(dg=np.array([[-cell1_x_growth, 0], [0, 0]]))  # y-growth for cell2

# Combine growth laws and assign
growth_scheme = HeterogeneousGrowth(cdata=rect.cdata, growth_classes={1: growth_cell1, 2: growth_cell2})

Run the growth problem simulation using the defined growth laws and visualize the results.

In [11]:
# Define the growth problem
mock_problem = GBVP(domain=rect, vform=plastic_vform, bc=bc, growth_scheme=growth_scheme)
mock_problem.run(tmin=0, tmax=1.01, dt=0.25, store_steps=True, linear_solver='mumps', line_search='bt') # 'bt': backtracking stabilize the convergence procedure

# Get the displacement field
displacement_field = mock_problem.solution_steps[1.0]  # Get results at final time step

# Visualize the displacement field
pl = pv.Plotter(shape=(1, 2), window_size=[1100, 400])
pl.subplot(0, 0)
visualize(mock_problem, 'domain', show_plot=False, plotter=pl, title='t=0')
pl.subplot(0, 1)
_, rect_cdata = rect.move(displacement_field)
visualize(rect_cdata, dict_values=rect.sub_domain_names, show_plot=False, plotter=pl, title='t=1')

pl.link_views()
pl.view_xy()
pl.show()

Run |■■■■■■■■■■■■■■■■■■■ | 99 %

Widget(value='<iframe src="http://localhost:36699/index.html?ui=P_0x7febf98144f0_10&reconnect=auto" class="pyv…

## Extract the strain-rate of each cells

Extract the strain fields for each cell and compute the mean strain values.

In [12]:
# Extract strain fields
strain = plastic_vform.get_strain(displacement_field, method='total')

# Convert to numpy, separate by subdomain, and calculate mean strain fields
strain_obj = SolutionExplorer(strain)
strain_fields = strain_obj.to_vertex_values()  # Convert to numpy array
tf_cell1 = strain_obj.get_vertex_positions()[:, 1] < 0.25  # Identify cell1 (no interface)
tf_cell2 = strain_obj.get_vertex_positions()[:, 1] > 0.25  # Identify cell2 (no interface)

# Compute mean strain fields
mean_cell1 = strain_fields[tf_cell1, :, :].mean(axis=0)
mean_cell2 = strain_fields[tf_cell2, :, :].mean(axis=0)

print("Moyenne pour cell1 :\n", np.around(mean_cell1, 2))
print("Moyenne pour cell2 :\n", np.around(mean_cell2, 2))

Moyenne pour cell1 :
 [[-0.05 -0.  ]
 [-0.    0.  ]]
Moyenne pour cell2 :
 [[-0.05  0.  ]
 [ 0.    0.  ]]


## Make the simulation of the cell growth

Use the computed mean strain fields as growth tensors for a follow-up experiment. 

In [13]:
# Define new growth laws using mean strain fields
growth_cell1 = ConstantGrowth(dg=mean_cell1)
growth_cell2 = ConstantGrowth(dg=mean_cell2)
growth_scheme = HeterogeneousGrowth(cdata=rect.cdata, growth_classes={1: growth_cell1, 2: growth_cell2})

# Define Potential Energy and Plastic Form
potential_energy = StVenantKirchoffPotential(young=0.0001, poisson=0)  # Stiffness parameters
plastic_vform = MorphoElasticForm(potential_energy=potential_energy, F_g=np.identity(2), source=[0, 0])

# Redefine and solve the growth problem
growth_problem = GBVP(domain=rect, vform=plastic_vform, bc=bc, growth_scheme=growth_scheme)
growth_problem.run(tmin=0, tmax=1.01, dt=0.25, store_steps=True, linear_solver='mumps', line_search='bt') # 'bt': backtracking stabilize the convergence procedure

# Visualize updated displacement field
displacement_field = growth_problem.solution_steps[1.0]

pl = pv.Plotter(shape=(1, 2), window_size=[1100, 400])
pl.subplot(0, 0)
visualize(growth_problem, 'domain', show_plot=False, plotter=pl, title='t=0')
pl.subplot(0, 1)
_, rect_cdata = rect.move(displacement_field)
visualize(rect_cdata, dict_values=rect.sub_domain_names, show_plot=False, plotter=pl, title='t=1')

pl.link_views()
pl.view_xy()
pl.show()

Run |■■■■■■■■■■■■■■■■■■■ | 99 %

Widget(value='<iframe src="http://localhost:36699/index.html?ui=P_0x7febf02bbdc0_13&reconnect=auto" class="pyv…

## Comparison mock data vs. simulation

In [14]:
# Visualize updated displacement field
real_displacement_field = mock_problem.solution_steps[1.0]
simul_displacement_field = growth_problem.solution_steps[1.0]

pl = pv.Plotter(shape=(1, 3), window_size=[1100, 400])
pl.subplot(0, 0)
visualize(growth_problem, 'domain', show_plot=False, plotter=pl, title='t=0')
pl.subplot(0, 1)
_, rect_cdata = rect.move(real_displacement_field)
visualize(rect_cdata, dict_values=rect.sub_domain_names, show_plot=False, plotter=pl, title='t=1\n(mock)')
pl.subplot(0, 2)
_, rect_cdata = rect.move(simul_displacement_field)
visualize(rect_cdata, dict_values=rect.sub_domain_names, show_plot=False, plotter=pl, title='t=1\n(simulated)')

pl.link_views()
pl.view_xy()
pl.show()

Widget(value='<iframe src="http://localhost:36699/index.html?ui=P_0x7febbc40a0d0_16&reconnect=auto" class="pyv…